# Model 1 - MLP Baseline (Member: Soham Amundkar, 230953340)

Human Activity Recognition from raw smartphone inertial signals (UCI HAR).

This notebook trains and evaluates the **Multi-Layer Perceptron baseline**: the
128 x 9 window is flattened to a 1,152-dimensional vector and passed through two
hidden layers (256 and 128 units) with batch normalisation, ReLU and dropout,
followed by a six-way classification head.

**Why this model is in the comparison.** It is the control. It encodes no temporal
and no spatial prior: permute the 128 timesteps before training and its accuracy is
unchanged, where the 1D-CNN's and the BiLSTM's would collapse. The gap between this
model and the other two is therefore a measurement of what architectural inductive
bias is worth on this task. Without it, "the CNN reached 0.92" is a number with
nothing to compare against.

**Second experiment.** The same architecture is also trained on the dataset's 561
hand-crafted time- and frequency-domain features, with identical optimiser settings.
That pair isolates the contribution of the engineered representation, and is the only
place in this project where representation is varied while architecture is held fixed.

Uses the shared `src/har_data.py` pipeline (subject-independent split, per-channel
standardisation fitted on training subjects only) and the shared `src/train.py` /
`src/evaluate.py` harness, so this model is trained and scored on byte-identical
data and metrics as the 1D-CNN and BiLSTM.


In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch

from har_data import set_seed, load_bundle, make_loaders, ACTIVITY_NAMES
from models.mlp import HARMlp
from train import train_model
from evaluate import evaluate_model, count_parameters, plot_confusion_matrix

DATA_ROOT = PROJECT_ROOT / "data" / "UCI HAR Dataset"
RESULTS_METRICS = PROJECT_ROOT / "results" / "metrics"
RESULTS_FIGURES = PROJECT_ROOT / "results" / "figures"
RESULTS_METRICS.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES.mkdir(parents=True, exist_ok=True)

SEED = 42
set_seed(SEED)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. Load data under the shared protocol

Subject-independent split: validation subjects (8, 16, 23, 29) are held out from the
21 training subjects, fixed in code so all three members train on identical data.
Standardisation statistics are fitted on the 17 remaining training subjects alone.

`with_features561=True` also loads the 561 hand-crafted feature vectors for the
second experiment below.


In [ ]:
bundle = load_bundle(DATA_ROOT, with_features561=True)
class_names = bundle["class_names"]

print()
print("val subjects held out:", bundle["val_subjects"])
print("raw window shape  :", bundle["X_train"].shape)
print("561-feature shape :", bundle["F_train"].shape)


### Leakage check

The claim that no subject appears in more than one partition is verified here rather
than taken from the dataset documentation. Because consecutive windows overlap by 50%,
a random window-level split would place near-duplicate windows on both sides and
inflate every figure in this notebook.


In [ ]:
tr = set(bundle["subj_train"].tolist())
va = set(bundle["subj_val"].tolist())
te = set(bundle["subj_test"].tolist())

print(f"train subjects: {len(tr)}  val: {len(va)}  test: {len(te)}")
print("train n val :", tr & va or "empty")
print("train n test:", tr & te or "empty")
print("val   n test:", va & te or "empty")
assert not (tr & va) and not (tr & te) and not (va & te)
print("\nno subject appears in more than one partition")


## 2. Experiment A - raw inertial signals

`channels_first=False` gives (N, 128, 9); `nn.Flatten()` collapses that to 1,152.


In [ ]:
train_loader, val_loader, test_loader = make_loaders(
    bundle, batch_size=128, channels_first=False)

model_raw = HARMlp(input_dim=128 * 9, n_classes=bundle["n_classes"])
print(model_raw)
print(f"\ntrainable parameters: {count_parameters(model_raw):,}")


In [ ]:
result_raw = train_model(
    model_raw, train_loader, val_loader,
    lr=1e-3, weight_decay=1e-4, max_epochs=100, patience=15,
)

print(f"\nbest epoch   : {result_raw['best_epoch']}")
print(f"best val loss: {result_raw['best_val_loss']:.4f}")
print(f"training time: {result_raw['training_time_sec']:.1f}s")


In [ ]:
metrics_raw = evaluate_model(model_raw, test_loader, device=result_raw["device"],
                             class_names=class_names)

print(f"test accuracy : {metrics_raw['accuracy']:.4f}")
print(f"macro F1      : {metrics_raw['macro_f1']:.4f}")
print()
print(metrics_raw["classification_report"])


## 3. Experiment B - 561 hand-crafted features

Same `HARMlp` class, same optimiser settings, same subjects, same seed. Only
`input_dim` changes, from 1,152 to 561. Any difference in result is therefore
attributable to the representation and not to a different training recipe.

The 561 features ship pre-normalised to [-1, 1] and are used as supplied.


In [ ]:
set_seed(SEED)  # re-seeded so the two experiments are independent

ftrain, fval, ftest = make_loaders(bundle, batch_size=128, features=True)

model_feat = HARMlp(input_dim=561, n_classes=bundle["n_classes"])
print(f"trainable parameters: {count_parameters(model_feat):,}")

result_feat = train_model(
    model_feat, ftrain, fval,
    lr=1e-3, weight_decay=1e-4, max_epochs=100, patience=15,
)

metrics_feat = evaluate_model(model_feat, ftest, device=result_feat["device"],
                              class_names=class_names)
print(f"\ntest accuracy : {metrics_feat['accuracy']:.4f}")
print(f"macro F1      : {metrics_feat['macro_f1']:.4f}")


## 4. Training curves and confusion matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (res, title) in zip(axes, [(result_raw, "MLP (raw signals)"),
                                   (result_feat, "MLP (561 features)")]):
    h = res["history"]
    ax.plot(h["train_loss"], label="train")
    ax.plot(h["val_loss"], label="val")
    ax.axvline(res["best_epoch"] - 1, color="grey", ls="--", label="best epoch")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title(title); ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "mlp_training_curves.png", dpi=150)
plt.show()


In [ ]:
for m, tag, title in [(metrics_raw, "mlp", "MLP (raw signals)"),
                      (metrics_feat, "mlp561", "MLP (561 features)")]:
    plot_confusion_matrix(m["confusion_matrix"], class_names,
                          save_path=RESULTS_FIGURES / f"{tag}_confusion_matrix.png",
                          title=f"{title} - normalised confusion matrix (test set)")
print("confusion matrices saved")


## 5. Error analysis

Two questions the comparison needs this model to answer:

1. **Where does the baseline lose?** If its errors are spread evenly across classes,
   the architectures above it are buying general capability. If they concentrate in
   one place, that place is what the inductive bias is actually fixing.
2. **Does the engineered representation fix the same errors, or different ones?**


In [ ]:
def error_profile(metrics, label):
    cm = np.asarray(metrics["confusion_matrix"], dtype=float)
    norm = cm / cm.sum(axis=1, keepdims=True)
    total_err = cm.sum() - np.trace(cm)
    print(f"--- {label} ---")
    print(f"total test errors: {int(total_err)} / {int(cm.sum())}")
    off = [(ACTIVITY_NAMES[i], ACTIVITY_NAMES[j], int(cm[i, j]), norm[i, j])
           for i in range(6) for j in range(6) if i != j and cm[i, j] > 0]
    off.sort(key=lambda t: -t[2])
    print("largest confusions (true -> predicted):")
    for t, p, n, frac in off[:5]:
        share = 100 * n / total_err
        print(f"  {t:<20} -> {p:<20} {n:4d}  ({frac:5.1%} of class, {share:4.1f}% of all errors)")
    static = cm[3:5, 3:5].sum() - np.trace(cm[3:5, 3:5])
    print(f"sitting/standing pair accounts for {100 * static / total_err:.1f}% of all errors\n")

error_profile(metrics_raw, "MLP (raw signals)")
error_profile(metrics_feat, "MLP (561 features)")


In [ ]:
# Per-class F1 side by side: which classes the representation helps.
rows = []
for c in class_names:
    a, b = metrics_raw["per_class_f1"][c], metrics_feat["per_class_f1"][c]
    rows.append((c, a, b, b - a))

print(f"{'class':<20} {'raw':>8} {'561':>8} {'delta':>8}")
for c, a, b, d in rows:
    print(f"{c:<20} {a:8.4f} {b:8.4f} {d:+8.4f}")

dynamic = [r for r in rows if r[0].startswith("WALKING")]
static = [r for r in rows if not r[0].startswith("WALKING")]
print(f"\nmean delta, dynamic classes: {np.mean([r[3] for r in dynamic]):+.4f}")
print(f"mean delta, static classes : {np.mean([r[3] for r in static]):+.4f}")


### Interpretation

_Fill this in from the output above once the cells have been run. The three things
worth stating, each of which follows directly from a number printed above:_

- _which confusion dominates the baseline's errors, and what share of all errors it is;_
- _whether the 561-feature representation improves the dynamic (walking) classes more
  than the static ones, and what that implies about what the engineered frequency-domain
  descriptors are capturing;_
- _whether the sitting/standing confusion survives both representations, which would
  indicate the limit is in the waist-mounted sensing geometry rather than in the model._


## 6. Save metrics for the comparison table


In [ ]:
for metrics, result, tag, label, rep in [
    (metrics_raw, result_raw, "mlp", "MLP (raw signals)", "raw 128x9 window"),
    (metrics_feat, result_feat, "mlp561", "MLP (561 features)", "561 hand-crafted features"),
]:
    payload = {
        "model": label,
        "owner": "Soham",
        "seed": SEED,
        "input_representation": rep,
        "val_subjects": list(bundle["val_subjects"]),
        "n_params": result["n_params"],
        "training_time_sec": result["training_time_sec"],
        "best_epoch": result["best_epoch"],
        "best_val_loss": result["best_val_loss"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_precision": metrics["macro_precision"],
        "test_macro_recall": metrics["macro_recall"],
        "test_macro_f1": metrics["macro_f1"],
        "per_class_f1": metrics["per_class_f1"],
        "confusion_matrix": metrics["confusion_matrix"],
    }
    out = RESULTS_METRICS / f"{tag}_metrics.json"
    out.write_text(json.dumps(payload, indent=2))
    print(f"saved -> {out.relative_to(PROJECT_ROOT)}")


## 7. Beyond the single run

Everything above is one run at seed 42, which cannot separate a one-point difference
from seed noise. Three further experiments, run from the command line:

```
python src/run_seeds.py --model mlp        # 5 seeds, raw signals
python src/run_seeds.py --model mlp561     # 5 seeds, 561 features
python src/run_seeds.py --summarize        # cross-model table + figure
python notebooks/mlp_ablation.py           # capacity and dropout sweep, both representations
python src/run_loso.py --model mlp         # leave-one-subject-out over all 30 volunteers
```

The ablation is what rules out the obvious objection to this baseline: that it loses to
the 1D-CNN only because it is too small. If accuracy barely moves between 64/32 and
512/256 hidden units, the model is not capacity-limited and the gap is about inductive
bias. Results land in `results/metrics/`.
